In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('../../src').resolve()))

In [ ]:
import download_data_mem
import processing
import utils
import pandas as pd
import numpy as np
import sys
import plot_radiosonde_ddu_map
import plot_radiosonde_linear 
import plot_radiosonde_skewt
import feature_engineering
import cluster_profile
import importlib
import matplotlib.pyplot as plt
from cluster_profile import ClusterProfiles, FeatureClusterProfiles
from datetime import date, datetime, timedelta
from pathlib import Path



importlib.reload(download_data_mem)
importlib.reload(processing)
importlib.reload(utils)
importlib.reload(plot_radiosonde_linear)
importlib.reload(plot_radiosonde_skewt)
importlib.reload(plot_radiosonde_ddu_map)
importlib.reload(cluster_profile)

In [ ]:
winter = download_data_mem.get_data(years=[2020,2021,2022,2023,2024,2025],months=[6,7,8],hours = [0]) #DJF / MAM / JJA / SON iwith 2 cons. year

In [ ]:
print("Winter:",len(winter))

In [ ]:
winter_int,winter_stats = utils.clean_and_interpolate(
    winter,
    z_grid=np.arange(50, 5000, 25),
    stitch=True,
    max_start_alt=200,   # strict : profil doit démarrer sous 200 m
    min_coverage=0.85    # doit atteindre 85 % de 5000 m ≈ 4300 m
) #try to 5km-6km

print("Winter:",len(winter_int))

In [ ]:
utils.plot_vertical_coverage(winter_int,var='t') #profils ne commencant pas à 0 ou terminant avant 5000m, ont des nan soit au debut soit à la fin ...

#essayer de check combien de profil comme ceux ci sont la suite du précédent...

In [ ]:
importlib.reload(cluster_profile)
# INIT
cp = FeatureClusterProfiles()

winter_int.pop(206)#strong outlier
winter_int.pop(488)#strong outlier
winter_int.pop(487)#strong outlier
winter_int.pop(486)#strong outlier

soundings = cp.filter_soundings(winter_int)
# BUILD MATRIX
df_features = cp.build_features_2(soundings=winter_int)

# CLEANING
X_clean = cp.handle_nan_2()

# NORMALISATION
X_norm = cp.normalize()
print(X_norm.shape)

In [ ]:
df_features.corr()

In [ ]:
cp.find_nb_of_components()

In [ ]:
X_red = cp.apply_pca(n_components=2)

In [ ]:
cp.find_k(X_red,max_k=10)

In [ ]:
labels = cp.fit(X_red,k=3)
labels = labels.astype(int)# for plotting purposes

In [ ]:
import numpy as np

plt.figure(figsize=(8,2))

for i in range(labels.min(), labels.max()+1):
    jitter = np.random.normal(0, 0.02, size=(labels == i).sum())
    
    plt.scatter(
        X_red[labels == i, 0],
        jitter,
        label=f'Cluster {i}',
        alpha=0.7
    )

plt.legend(title='Clusters')
plt.xlabel('PCA Component 1')
plt.yticks([])
plt.title('K-means Clusters in 1D PCA Space')
plt.show()

In [ ]:
plt.figure(figsize=(8,6))

for i in range(labels.min(), labels.max()+1):
    plt.scatter(
        X_red[labels == i, 0],
        X_red[labels == i, 1],
        label=f'Cluster {i}'
    )

plt.legend(title='Clusters')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.title('K-means Clusters in PCA Space')
plt.show()

In [ ]:
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
pca = PCA()
pca.fit(X_norm)

# ============================================================
# BLOC 1 — Variance expliquée par composante
# ============================================================
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1, len(pca.explained_variance_ratio_) + 1),
       pca.explained_variance_ratio_,
       label='Variance par composante')
ax.plot(range(1, len(pca.explained_variance_ratio_) + 1),
        pca.explained_variance_ratio_.cumsum(),
        color='red', marker='o', label='Variance cumulée')
ax.axhline(0.9, color='grey', linestyle='--', label='Seuil 90%')
ax.set_xlabel('Composante PCA')
ax.set_ylabel('Variance expliquée')
ax.set_title('Variance expliquée par composante PCA')
ax.legend()
plt.tight_layout()
plt.show()

# Affichez les valeurs numériques — regardez PC1
for i, v in enumerate(pca.explained_variance_ratio_):
    print(f"PC{i+1}: {v:.3f}  (cumulé: {pca.explained_variance_ratio_[:i+1].sum():.3f})")


# BLOC 2 — Projection PC1 vs PC2 colorée par cluster k=3-4

# Labels du clustering d'avant k=3/4/.. 
labels = cp.labels  # adaptez selon votre attribut

# Re-fit PCA à 2 composantes pour la visualisation
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_norm)

fig, ax = plt.subplots(figsize=(7, 6))
scatter = ax.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1],
                     c=labels, cmap='Set1', alpha=0.6, s=40)
plt.colorbar(scatter, ax=ax, label='Cluster')
ax.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title(f'Projection PCA 2D — clusters k={len(np.unique(labels))}')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# BLOC 3 — Silhouette score : n_components=1..8, k=3 fixé
n_max = min(8, X_norm.shape[1])  # pas plus que le nb de features
scores = []

for n in range(1, n_max + 1):
    X_pca_n = PCA(n_components=n).fit_transform(X_norm)
    km = KMeans(n_clusters=len(np.unique(labels)), n_init=20, random_state=42).fit(X_pca_n)
    s = silhouette_score(X_pca_n, km.labels_)
    scores.append(s)
    print(f"n={n}: silhouette={s:.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, n_max + 1), scores, marker='o', color='steelblue')
ax.axvline(np.argmax(scores) + 1, color='red', linestyle='--',
           label=f'Optimal n={np.argmax(scores)+1}')
ax.set_xlabel('Nombre de composantes PCA')
ax.set_ylabel('Silhouette score (k=3)')
ax.set_title('Silhouette vs dimension PCA')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nMeilleur n : {np.argmax(scores)+1} (silhouette={max(scores):.4f})")

In [ ]:
# Que contient PC1 ? Quelles features le composent ?
pca_full = PCA()
pca_full.fit(X_norm)

feature_names = cp.df_features.columns.tolist()

# Loadings PC1 et PC2
for pc_idx, pc_name in [(0, 'PC1'), (1, 'PC2'),(2, 'PC3'),(3, 'PC4'),(4, 'PC5')]:
    loadings = pca_full.components_[pc_idx]
    sorted_idx = np.argsort(np.abs(loadings))[::-1]
    print(f"\n--- {pc_name} ({pca_full.explained_variance_ratio_[pc_idx]*100:.1f}%) ---")
    for i in sorted_idx[:8]:  # top 8 features
        print(f"  {feature_names[i]:25s}: {loadings[i]:+.3f}")

PC1 : vent direction et intensité -> katabatique vs synoptique
PC2 : stabilité thermique et humidité de surface

PC2 (+) : stable, sec, inversion forte
                         ▲
                         │
                         │
  PC1 (−)                │                    PC1 (+)
  synoptique ────────────┼─────────────── katabatique
  flux ouest, jet fort   │         flux est, jet haut/modéré
                         │
                         ▼
                    PC2 (−) : instable, humide, cisaillement fort


katabatique =? est, Et synoptique =? ouest        voir avec prof. 

PC1:  [−−− C0 katabatique −−−|−−− C1 transition −−−|−−− C2 synoptique −−−]

In [ ]:
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import numpy as np

pca_2d = PCA(n_components=2)
X_2d = pca_2d.fit_transform(X_norm)

# Tester k=3,4,5 dans l'espace 2D
for k in [3, 4, 5]:
    km = KMeans(n_clusters=k, n_init=50, random_state=42)
    labels_test = km.fit_predict(X_2d)
    s = silhouette_score(X_2d, labels_test)
    print(f"k={k}, n=2 : silhouette={s:.4f}")

# Visualisation des clusters dans PC1/PC2
import matplotlib.pyplot as plt

km_best = KMeans(n_clusters=4, n_init=50, random_state=42)
labels_2d = km_best.fit_predict(X_2d)

fig, ax = plt.subplots(figsize=(8, 7))
scatter = ax.scatter(X_2d[:, 0], X_2d[:, 1],
                     c=labels_2d, cmap='Set1', alpha=0.6, s=40)

# Centroides
centers = km_best.cluster_centers_
ax.scatter(centers[:, 0], centers[:, 1], 
           c='black', marker='X', s=200, zorder=5, label='Centroides')

# Annotations axes
ax.axhline(0, color='grey', lw=0.5, linestyle='--')
ax.axvline(0, color='grey', lw=0.5, linestyle='--')

ax.set_xlabel(f'PC1 — Katabatique↔Synoptique ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 — Stable/Sec↔Instable/Humide ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('Clustering dans espace PC1–PC2')

# Annotations quadrants
ax.text(-3.5,  3.5, 'Synoptique\nstable sec',   fontsize=9, color='grey')
ax.text( 1.5,  3.5, 'Katabatique\nstable sec',    fontsize=9, color='grey')
ax.text(-3.5, -3.5, 'Synoptique\nhumide',        fontsize=9, color='grey')
ax.text( 1.5, -3.5, 'Katabatique\nhumide',         fontsize=9, color='grey')

plt.colorbar(scatter, ax=ax, label='Cluster')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import gaussian_kde
import matplotlib.pyplot as plt
import numpy as np

# KDE sur le scatter plot
xy = X_2d.T
kde = gaussian_kde(xy, bw_method=0.3)
xmin, xmax = X_2d[:,0].min()-0.5, X_2d[:,0].max()+0.5
ymin, ymax = X_2d[:,1].min()-0.5, X_2d[:,1].max()+0.5
xx, yy = np.mgrid[xmin:xmax:100j, ymin:ymax:100j]
positions = np.vstack([xx.ravel(), yy.ravel()])
density = kde(positions).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(8, 7))
ax.contourf(xx, yy, density, levels=15, cmap='Blues')
ax.contour(xx, yy, density, levels=15, colors='navy', linewidths=0.5)
ax.scatter(X_2d[:,0], X_2d[:,1], c='red', s=10, alpha=0.4)
ax.set_xlabel(f'PC1 — Katabatique↔Synoptique')
ax.set_ylabel(f'PC2 — Stable/Sec↔Humide')
ax.set_title('Densité KDE dans espace PC1–PC2')
plt.tight_layout()
plt.show()

# Regardez si la densité a des "vallées" entre les clusters actuels
fig, ax = plt.subplots(figsize=(8, 7))
ax.contourf(xx, yy, density, levels=15, cmap='Blues')
ax.contour(xx, yy, density, levels=15, colors='navy', linewidths=0.5)

# Superposez vos labels k=3/4 actuels (depuis n=1)
colors = ['red', 'green', 'orange','purple']  

print(np.unique(labels))
for k in range(4):
    mask = cp.labels == k
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               c=colors[k], s=15, alpha=0.6, label=f'C{k}')

ax.legend()
plt.show()

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

# KMeans directement sur PC1+PC2
km_2d = KMeans(n_clusters=4, n_init=100, random_state=42)
labels_2d = km_2d.fit_predict(X_2d)

s_2d = silhouette_score(X_2d, labels_2d)
print(f"Silhouette KMeans 2D, k=4 : {s_2d:.4f}")

# Comparer avec votre solution actuelle k=3 sur PC1
ari = adjusted_rand_score(cp.labels, labels_2d)
print(f"ARI vs solution PC1 k=3 : {ari:.3f}")

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors = ['red', 'green', 'orange', 'purple','blue']

# Gauche : votre solution actuelle k=3 PC1
ax = axes[0]
ax.contourf(xx, yy, density, levels=10, cmap='Blues', alpha=0.3)
for k in range(3):
    mask = cp.labels == k
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               c=colors[k], s=15, alpha=0.6, label=f'C{k} (n={mask.sum()})')
ax.set_title('KMeans k=3 sur PC1 (solution actuelle)')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.legend()

# Droite : nouvelle solution k=4 sur PC1+PC2
ax = axes[1]
ax.contourf(xx, yy, density, levels=10, cmap='Blues', alpha=0.3)
for k in range(4):
    mask = labels_2d == k
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               c=colors[k], s=15, alpha=0.6, label=f'C{k} (n={mask.sum()})')
ax.set_title('KMeans k=4 sur PC1+PC2 (nouvelle solution)')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.legend()

plt.tight_layout()
plt.show()

# Tester aussi k=3 en 2D pour comparaison complète
for k in [3, 4, 5]:
    km = KMeans(n_clusters=k, n_init=100, random_state=42)
    labels = km.fit_predict(X_2d)
    s = silhouette_score(X_2d, labels)
    print(f"k={k} sur PC1+PC2 : silhouette={s:.4f}")

In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
import numpy as np

results = {}
for k in [3, 4, 5]:
    best_gmm = None
    best_bic = np.inf
    # Multiple restarts car GMM sensible à l'initialisation
    for seed in range(30):
        gmm = GaussianMixture(n_components=k,
                              covariance_type='full',
                              n_init=1,
                              random_state=seed)
        gmm.fit(X_2d)
        if gmm.bic(X_2d) < best_bic:
            best_bic = gmm.bic(X_2d)
            best_gmm = gmm
    
    labels_gmm = best_gmm.predict(X_2d)
    s = silhouette_score(X_2d, labels_gmm)
    results[k] = {'gmm': best_gmm, 'labels': labels_gmm,
                  'bic': best_bic, 'silhouette': s}
    print(f"k={k}: silhouette={s:.4f}, BIC={best_bic:.1f}")

# Visualisation du meilleur
best_k = min(results, key=lambda k: results[k]['bic'])
print(f"\nMeilleur k selon BIC : {best_k}")

labels_best = results[best_k]['labels']
fig, ax = plt.subplots(figsize=(8, 7))
ax.contourf(xx, yy, density, levels=10, cmap='Blues', alpha=0.4)

colors = ['red', 'green', 'orange', 'purple', 'brown']
for k in range(best_k):
    mask = labels_best == k
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               c=colors[k], s=20, alpha=0.7, label=f'C{k} (n={mask.sum()})')

# Ellipses de confiance GMM
from matplotlib.patches import Ellipse
import matplotlib.transforms as transforms

def plot_gmm_ellipse(ax, mean, cov, color, n_std=2.0):
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    angle = np.degrees(np.arctan2(*vecs[:,0][::-1]))
    width, height = 2 * n_std * np.sqrt(vals)
    ellipse = Ellipse(xy=mean, width=width, height=height,
                      angle=angle, edgecolor=color,
                      fc='None', lw=2, linestyle='--')
    ax.add_patch(ellipse)

gmm_best = results[best_k]['gmm']
for i, (mean, cov) in enumerate(zip(gmm_best.means_,
                                     gmm_best.covariances_)):
    plot_gmm_ellipse(ax, mean, cov, colors[i])

ax.set_xlabel(f'PC1 — Katabatique↔Synoptique ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 — Stable/Sec↔Humide ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title(f'GMM k={best_k} — ellipses de confiance 2σ')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Extrayez les dates par cluster
for k in range(3):
    idx = np.where(cp.labels == k)[0]
    dates = [winter_int[i]['header']['date'] for i in idx]
    print(f"\nCluster {k} : {len(dates)} soundings")
    print(dates[:5])

In [ ]:
summary = cp.cluster_summary()
print(summary)

In [ ]:
cp.silhouette_hierarchical(X_red)

In [ ]:
cp.plot_dendrogram(X_red,show_cut_at_k=4)

In [ ]:
labels = cp.fit_hierarchical(X_red, k=3)

In [ ]:
# Visualisation PCA (marche pour les deux)
cp.plot_pca_scatter(title="Clustering hiérarchique — hiver")

In [ ]:
cp.plot_cluster_minipages(winter_int,z_grid=np.arange(50, 5000, 25)[1:],quantiles=(0.25,0.75),ylim=(0, 5000))

In [ ]:
print(cp.cluster_summary_full(winter_int))

In [ ]:
cp.diagnose_clusters(winter_int)

In [ ]:
cp.plot_features()

In [ ]:
def compute_mean_wind_by_cluster(soundings, labels, z_max=1000):

    import numpy as np

    clusters = np.unique(labels)

    results = {}

    for k in clusters:

        u_all = []
        v_all = []

        for i, s in enumerate(soundings):

            if labels[i] != k:
                continue

            df = s['data'].copy()

            if not all(var in df.columns for var in ['ff', 'dd', 'altitude']):
                continue

            # reconstruction u/v
            ff = df['ff'].values
            dd = np.deg2rad(df['dd'].values)

            u = -ff * np.sin(dd)
            v = -ff * np.cos(dd)

            z = df['altitude'].values
            mask = z <= z_max

            if np.sum(mask) < 3:
                continue

            u_all.extend(u[mask])
            v_all.extend(v[mask])

        if len(u_all) == 0:
            results[k] = (np.nan, np.nan, np.nan)
            continue

        u_mean = np.nanmean(u_all)
        v_mean = np.nanmean(v_all)
        wind_speed = np.nanmean(np.sqrt(np.array(u_all)**2 + np.array(v_all)**2))

        results[k] = (u_mean, v_mean, wind_speed)

    return results

In [ ]:
wind_stats = compute_mean_wind_by_cluster(winter_int, labels)

for k, (u, v, ws) in wind_stats.items():
    print(f"\nCluster {k}")
    print(f"Mean u: {u:.2f} m/s")
    print(f"Mean v: {v:.2f} m/s")
    print(f"Mean wind speed: {ws:.2f} m/s")